# **Teen Mental Health Risk Analyzer — Social Media Impact Study**

This notebook analyzes the impact of social media on teenager mental health using machine learning models. Two predictive models are implemented:

   1. Mental Health Risk Prediction Model: predicts individual teenager risk levels (Low, Medium, or High) based on social media usage patterns, sleep habits, platform behavior, and lifestyle metrics.
   2. Mental Health Index Prediction Model: estimates a continuous mental health score for each teenager based on anxiety, depression, and stress indicators combined with daily screen time and demographic features.

# **Stage 1: Retrieving and loading the dataset**

As the first step, we load the dataset and provide some initial overview on the data through shape, columns and info methods

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [14]:
# Load the dataset
file_path = '../dataset/Teen_Mental_Health_Dataset.csv'
data = pd.read_csv(file_path);

In [9]:
print(data.head())

   age  gender  daily_social_media_hours platform_usage  sleep_hours  \
0   14    male                       7.9      Instagram          7.4   
1   19  female                       1.9         TikTok          8.0   
2   17  female                       1.3      Instagram          7.6   
3   15    male                       7.4         TikTok          6.9   
4   15  female                       4.7           Both          4.9   

   screen_time_before_sleep  academic_performance  physical_activity  \
0                       2.9                  3.01                1.5   
1                       2.9                  3.22                0.8   
2                       0.5                  3.92                0.0   
3                       1.6                  3.48                0.8   
4                       3.0                  2.37                1.4   

  social_interaction_level  stress_level  anxiety_level  addiction_level  \
0                      low             2              2   

In [11]:
print(f"Shape of the data (rows, columns): {data.shape}")
print(f"Columns in data: {list(data.columns)}\n")

data.info()

Shape of the data (rows, columns): (1200, 13)
Columns in data: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   object 
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   object 
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_lev

#### Initial Data Quality Assessment

- The dataset contains a mix of numerical and categorical features, so encoding will be required for columns such as `gender`, `platform`, and `usage_frequency`.
- No missing values were found across any column.
- Minor inconsistencies in categorical labels (e.g. casing differences) were identified and will be addressed during cleaning.
- The dataset is structured and ready for preprocessing.

This confirms the data is complete and suitable for analysis after encoding and label standardization.

## **Stage 2: Data Preparation**
In this section, we prepare the dataset for teenager mental health risk prediction.
This includes:
1. Data Cleansing : removing duplicates, inconsistent categorical labels, and invalid or out-of-range values.
2. Data Transformation : feature engineering (mental health index, risk labels, sleep deficit), categorical encoding, and feature scaling.

### 2.1 Data Cleansing

In [17]:
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = data.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates found.")

Number of duplicate rows: 0
No duplicates found.


No duplicates were found in the dataset.

In [19]:
missing_values = data.isnull().sum();
print(f'Number of missing values in each column:\n{missing_values}')

Number of missing values in each column:
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64


No missing values were found in dataset, indicating completeness and suitability for modeling

In [31]:
invalid_age = data[(data['age'] < 10) | (data['age'] > 20)].shape[0]
invalid_daily_hours = data[(data['daily_social_media_hours'] < 0) | (data['daily_social_media_hours'] > 16)].shape[0]
invalid_sleep_hours = data[(data['sleep_hours'] <= 0) | (data['sleep_hours'] > 14)].shape[0]
invalid_anxiety = data[(data['anxiety_level'] < 0) | (data['anxiety_level'] > 10)].shape[0]
invalid_depression = data[(data['depression_label'] < 0) | (data['depression_label'] > 1)].shape[0]
invalid_stress = data[(data['stress_level'] < 0) | (data['stress_level'] > 10)].shape[0]
invalid_academic = data[(data['academic_performance'] < 0) | (data['academic_performance'] > 100)].shape[0]
invalid_screen_before_sleep = data[(data['screen_time_before_sleep'] < 0) | (data['screen_time_before_sleep'] > 12)].shape[0]
invalid_addiction_level = data[(data['addiction_level'] < 0) | (data['addiction_level'] > 10)].shape[0]

print(f"Invalid Age values: {invalid_age}")
print(f"Invalid Daily Usage Hours: {invalid_daily_hours}")
print(f"Invalid Sleep Hours: {invalid_sleep_hours}")
print(f"Invalid Anxiety Score: {invalid_anxiety}")
print(f"Invalid Depression Label: {invalid_depression}")
print(f"Invalid Stress Score: {invalid_stress}")
print(f"Invalid Academic Performance: {invalid_academic}")
print(f"Invalid Screen Time Before Sleep: {invalid_screen_before_sleep}")
print(f"Invalid Addiction Level: {invalid_addiction_level}")

Invalid Age values: 0
Invalid Daily Usage Hours: 0
Invalid Sleep Hours: 0
Invalid Anxiety Score: 0
Invalid Depression Label: 0
Invalid Stress Score: 0
Invalid Academic Performance: 0
Invalid Screen Time Before Sleep: 0
Invalid Addiction Level: 0


['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']
